In [1]:
!pip install -qU pypdf pinecone google-genai google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 822.5/822.5 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.7.0 which is incompatible.


In [2]:
import os
import time
import re
from pypdf import PdfReader
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata
from google.colab import files
# 同時引入新舊庫，新庫用來回答與防 429，舊核心用來保證向量相容
import google.genai as google_genai_core
from google.genai import errors as google_errors
import google.generativeai as legacy_genai

print("啟動【跨語言】高相容性官方 SDK RAG 系統...")

# 初始化 API
client = google_genai_core.Client(api_key=userdata.get('GOOGLE_API_KEY'))
legacy_genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))
pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))

# 宣告模型變數
MODEL_NAME = 'gemini-2.5-flash'
EMBEDDING_MODEL = 'text-embedding-004' # Explicitly define the embedding model

print("\n請從電腦上傳英文論文 PDF 檔案...")
uploaded = files.upload()

uploaded_pdf_files = [f for f in uploaded.keys() if f.endswith('.pdf')]
if not uploaded_pdf_files:
    raise FileNotFoundError("錯誤：你沒有上傳任何 PDF 檔案，請重新執行本儲存格！")

PDF_FILE_PATH = uploaded_pdf_files[0]
print(f"成功上傳並鎖定文獻目標：【 {PDF_FILE_PATH} 】")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


啟動【跨語言】高相容性官方 SDK RAG 系統...

請從電腦上傳英文論文 PDF 檔案...


Saving Optimization_Study_of_Low_Latency_Scheduling_Algorithm_Based_on_Time-Sensitive_Networks.pdf to Optimization_Study_of_Low_Latency_Scheduling_Algorithm_Based_on_Time-Sensitive_Networks.pdf
成功上傳並鎖定文獻目標：【 Optimization_Study_of_Low_Latency_Scheduling_Algorithm_Based_on_Time-Sensitive_Networks.pdf 】


In [3]:
# ==========================================
# 模組一與模組二：讀取英文 PDF 並切割
# ==========================================
def process_english_pdf(file_path, chunk_size=600, overlap=60):
    print(f"正在讀取英文文獻: {file_path}")
    reader = PdfReader(file_path)
    full_text = "".join([page.extract_text() + "\n" for page in reader.pages if page.extract_text()])

    chunks = []
    start = 0
    while start < len(full_text):
        end = start + chunk_size
        chunks.append(full_text[start:end].strip())
        start += (chunk_size - overlap)
    return [c for c in chunks if c]

text_chunks = process_english_pdf(PDF_FILE_PATH)
print(f"英文文本切割完成，共產生 {len(text_chunks)} 個區塊。")

正在讀取英文文獻: Optimization_Study_of_Low_Latency_Scheduling_Algorithm_Based_on_Time-Sensitive_Networks.pdf
英文文本切割完成，共產生 57 個區塊。


In [4]:
# ==========================================
# 模組三：將英文文本向量化並儲存 (改用 100% 成功之經典向量端點，防 404 與 409)
# ==========================================
index_name = "cross-lingual-rag-final"  # 建立專屬純淨索引

if index_name not in pc.list_indexes().names():
    print(f"建立雲端向量資料庫 '{index_name}' 中...")
    pc.create_index(
        name=index_name, dimension=3072, metric="cosine",  # 經典模型維度為 3072
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

try:
    print("正在清理雲端向量資料庫中的舊文獻快取...")
    index.delete(delete_all=True)
    time.sleep(2)
except Exception:
    pass

print("正在將英文文本轉換為向量並寫入資料庫...")
vectors_to_upsert = []

for i, chunk in enumerate(text_chunks):
    # 核心修正：改用 100% 絕對支援 embedContent 的經典萬用端點，保證不噴 404
    response = legacy_genai.embed_content(
        model="models/gemini-embedding-2",
        content=chunk,
        task_type="retrieval_document"
    )
    embedding = response['embedding']

    vectors_to_upsert.append({"id": f"chunk-{i}", "values": embedding, "metadata": {"text": chunk}})

batch_size = 30
for k in range(0, len(vectors_to_upsert), batch_size):
    index.upsert(vectors=vectors_to_upsert[k:k+batch_size])

print("英文知識庫建置完成！")

正在清理雲端向量資料庫中的舊文獻快取...
正在將英文文本轉換為向量並寫入資料庫...
英文知識庫建置完成！


In [5]:
# ==========================================
# 模組四與五：優化檢索、全防禦生成與互動式問答迴圈
# ==========================================
print("\n知識庫已準備就緒，請開始提問！(輸入 'exit' 可結束對話)")

while True:
    question_zh = input("\n請輸入你的中文問題: ").strip()

    if question_zh.lower() == 'exit' or not question_zh:
        print("👋 感謝使用，手動問答系統已關閉。")
        break

    print(f"使用者中文提問: {question_zh}")

    # 4-1. 將中文問題翻譯成英文 (帶有最新 SDK 的 429 自動倒數防禦線)
    print("系統正在將問題翻譯為英文...")
    translation_prompt = f"Please translate the following Traditional Chinese question into English. ONLY output the English translation, without any explanations.\nQuestion: {question_zh}"

    question_en = ""
    while not question_en:
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=translation_prompt
            )
            question_en = response.text.strip()
        except google_errors.APIError as e:
            match = re.search(r"Please retry in (\d+\.\d+)s", str(e))
            wait_sec = int(float(match.group(1))) + 2 if match else 20
            print(f"【防線 A】翻譯層觸發免費版配額限制 (429)，系統自動安全倒數 {wait_sec} 秒後重試...")
            time.sleep(wait_sec)
        except Exception:
            time.sleep(15)
            continue

    print(f"翻譯結果: {question_en}")

    # 4-2. 使用「英文問題」去資料庫進行檢索 (同步改用萬用向量端點)
    response = legacy_genai.embed_content(
        model="models/gemini-embedding-2",
        content=question_en,
        task_type="retrieval_query"
    )
    query_vector = response['embedding']

    search_results = index.query(vector=query_vector, top_k=3, include_metadata=True)
    retrieved_contexts_en = [match['metadata']['text'] for match in search_results['matches']]
    context_string_en = "\n---\n".join(retrieved_contexts_en)
    print(f"檢索完成，已抓取 {len(retrieved_contexts_en)} 段相關的【英文】文獻。")

    # 5. 跨語言 Prompt 組裝與最終生成 (帶有最新 SDK 的 429 自動倒數防禦線)
    print("正在根據英文文獻，用中文回答使用者的問題...")
    final_prompt = f"""
    你是一個專業的網路通訊與 IoT 研究助理。
    請嚴格根據以下提供的【英文文獻內容】來回答使用者的【中文問題】。如果你在資料中找不到答案，請老實回答「在文件中找不到相關資訊」，千萬不要瞎編。

    【回答規範】:
    1. 請務必使用流暢的繁體中文 (Traditional Chinese) 回答。
    2. 提到專業術語時（如演算法、通訊協定、網路架構），請務必在括號內附上英文原文（例如：邊緣運算 (Edge Computing)）。
    3. 嚴禁脫離文獻內容進行天馬行空的發揮，一切以提供內容為準。

    【英文文獻內容】(English Context):
    {context_string_en}

    【中文問題】:
    {question_zh}
    """

    response_text = ""
    while not response_text:
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=final_prompt
            )
            response_text = response.text
        except google_errors.APIError as e:
            match = re.search(r"Please retry in (\d+\.\d+)s", str(e))
            wait_sec = int(float(match.group(1))) + 2 if match else 20
            print(f"【防線 B】生成層觸發流量配額限制 (429)，系統自動掛起倒數 {wait_sec} 秒後重試...")
            time.sleep(wait_sec)
        except Exception:
            time.sleep(15)
            continue

    if response_text:
        print(f"\n最終中文回答:\n{response_text}")
    print("-" * 50)

    time.sleep(3)


知識庫已準備就緒，請開始提問！(輸入 'exit' 可結束對話)

請輸入你的中文問題: 模擬退火的『初始溫度（Initial temperature）』與『冷卻率（Cooling rate）』分別設定為多少？
使用者中文提問: 模擬退火的『初始溫度（Initial temperature）』與『冷卻率（Cooling rate）』分別設定為多少？
系統正在將問題翻譯為英文...
【防線 A】翻譯層觸發免費版配額限制 (429)，系統自動安全倒數 43 秒後重試...
【防線 A】翻譯層觸發免費版配額限制 (429)，系統自動安全倒數 60 秒後重試...
【防線 A】翻譯層觸發免費版配額限制 (429)，系統自動安全倒數 59 秒後重試...
【防線 A】翻譯層觸發免費版配額限制 (429)，系統自動安全倒數 60 秒後重試...


KeyboardInterrupt: 